# L03 — Conceptual Modeling: The Primary Care Clinic

**Module**: M02 | **Chapter**: 3 | **Lecture**: L03

## Learning Objectives
By the end of this notebook you will be able to:
1. Apply the entity–resource–event–state framework to a multi-stage healthcare system.
2. Identify endogenous vs. exogenous variables and justify boundary choices.
3. Distinguish events from activities and explain why both are needed.
4. Produce a complete conceptual model document with explicit assumptions.

---
> **Think → Trace → Code → Experiment → Interpret → Communicate**

No code to run yet — this is a structured thinking lab. The Python cells serve as scaffolding for your written analysis.
---

## System Description

A primary care clinic operates 8 hours per day. Walk-in patients arrive, register at the front desk, wait for a triage nurse, then wait for an available exam room where they see a physician.

**Study question**: *How many triage nurses minimise mean patient time in clinic while keeping nurse utilisation above 70%?*

We will build the complete conceptual model in stages.

## 1. System Boundaries

The boundary choice determines scope. Three options for this clinic:

In [ ]:
boundary_options = {
    'Narrow': {
        'inside':  ['Waiting room', 'Registration desk', 'Triage bays', 'Exam rooms'],
        'outside': ['Scheduling system', 'Pharmacy', 'Lab'],
        'answers': 'How long do patients wait? Which stage is the bottleneck?',
    },
    'Medium': {
        'inside':  ['Narrow scope +', 'Appointment scheduling', 'Lab order processing'],
        'outside': ['Pharmacy', 'Hospital bed allocation'],
        'answers': 'How does scheduling policy affect walk-in wait?',
    },
    'Wide': {
        'inside':  ['Medium scope +', 'Pharmacy', 'Lab'],
        'outside': ['Hospital network', 'Insurance/payer systems'],
        'answers': 'How does clinic congestion affect downstream hospital occupancy?',
    },
}

for name, b in boundary_options.items():
    print(f"{name} boundary")
    print(f"  Inside  : {', '.join(b['inside'])}")
    print(f"  Outside : {b['outside']}")
    print(f"  Answers : {b['answers']}")
    print()

In [ ]:
# Exercise 1: Justify the boundary for our study question
# Fill in your answer as a string

boundary_choice = "?"   # 'Narrow', 'Medium', or 'Wide'
justification = """
The study question asks about nurse staffing and patient wait times.
We need [your reasoning here].
We exclude [what and why].
"""

print(f"Chosen boundary: {boundary_choice}")
print(f"Justification  : {justification}")

## 2. Entities and Attributes

Fill in the entity table for the narrow-boundary clinic model.

In [ ]:
import pandas as pd

entity_table = pd.DataFrame([
    {'Entity': 'Patient',
     'Attributes': 'arrival_time, acuity (1–5), appointment_type (walk-in/scheduled)',
     'Entry point': 'Clinic entrance',
     'Exit point': 'After physician exam'},
    {'Entity': '?',
     'Attributes': '?',
     'Entry point': '?',
     'Exit point': '?'},
])

print(entity_table.to_string(index=False))
print("\nHint: are there other entity types, or are all entities 'patients'?")

## 3. Resources

A resource has: capacity, service time distribution, and service discipline.

In [ ]:
resource_table = pd.DataFrame([
    {'Resource': 'Registration clerk',
     'Capacity': 1,
     'Service time': 'Exp(mean=3 min)',
     'Discipline': 'FCFS'},
    {'Resource': 'Triage nurse',
     'Capacity': 'c_n (decision variable)',
     'Service time': 'Exp(mean=8 min)',
     'Discipline': 'FCFS'},
    {'Resource': 'Exam room (physician)',
     'Capacity': 2,
     'Service time': 'Exp(mean=15 min)',
     'Discipline': 'FCFS'},
])

print(resource_table.to_string(index=False))
print()

# Exercise: for each resource, answer:
questions = [
    "Which resource is the bottleneck when c_n=1? When c_n=2?",
    "What would change the service discipline from FCFS to Priority?",
    "If exam rooms = 2, what is the max system throughput in patients/hour?",
]
for i, q in enumerate(questions, 1):
    print(f"Q{i}: {q}")
    print(f"A{i}: ?")
    print()

## 4. Events and Activities

In [ ]:
event_table = pd.DataFrame([
    {'Event': 'Patient arrival',
     'What changes': 'n_waiting_reg += 1; if reg free: start registration',
     'Triggers': 'Registration-begin (if server idle)'},
    {'Event': 'Registration-begin',
     'What changes': 'n_waiting_reg -= 1; clerk becomes busy',
     'Triggers': 'Registration-end event'},
    {'Event': 'Registration-end',
     'What changes': 'Clerk idle; patient moves to triage queue',
     'Triggers': 'Triage-begin (if nurse free); next reg-begin (if queue non-empty)'},
    {'Event': 'Triage-begin',      'What changes': '?', 'Triggers': '?'},
    {'Event': 'Triage-end',        'What changes': '?', 'Triggers': '?'},
    {'Event': 'Exam-begin',        'What changes': '?', 'Triggers': '?'},
    {'Event': 'Patient departure',  'What changes': '?', 'Triggers': '?'},
])

print(event_table.to_string(index=False))
print()

activities = pd.DataFrame([
    {'Activity': 'Interarrival time', 'Duration': 'Exp(mean=12 min)', 'Between events': 'Consecutive arrivals'},
    {'Activity': 'Registration service', 'Duration': 'Exp(3 min)', 'Between events': 'Reg-begin to Reg-end'},
    {'Activity': 'Triage service',  'Duration': '?',             'Between events': '?'},
    {'Activity': 'Physician exam',   'Duration': '?',             'Between events': '?'},
])
print(activities.to_string(index=False))

## 5. State Variables and Performance Measures

In [ ]:
state_vars = [
    'n_reg   : number in registration queue + being registered',
    'n_triage: number in triage queue + being triaged',
    'n_exam  : number in exam queue + in exam room',
    'b_reg   : 0/1 registration clerk busy',
    'b_nurse : 0..c_n nurses currently busy',
    'b_exam  : 0..2 exam rooms currently occupied',
]

performance_measures = {
    'W_total': 'Mean patient time in clinic (arrival to departure)',
    'W_triage': 'Mean wait in triage queue',
    'rho_nurse': 'Triage nurse utilisation = fraction of time busy',
    'rho_exam': 'Physician/exam room utilisation',
    'L_max': '?',   # what would this be?
    'P_long_wait': '?',   # probability of waiting > some threshold
}

print("State variables (minimal set to resume from any snapshot):")
for v in state_vars:
    print(f"  {v}")

print("\nPerformance measures:")
for k, v in performance_measures.items():
    print(f"  {k:20s}: {v}")

## 6. Throughput Bound — Before Writing Any Code

Before simulation, compute a simple analytical bound on maximum throughput.
This tells you whether the parameters make sense.

In [ ]:
# Arrival rate
lam = 5.0   # patients/hour (= 1 every 12 min)

# Service rates (patients/hour per resource unit)
mu_reg   = 60.0 / 3.0    # 20 patients/hour per clerk
mu_nurse = 60.0 / 8.0    # 7.5 patients/hour per nurse
mu_exam  = 60.0 / 15.0   # 4 patients/hour per room

# Capacity
c_reg  = 1
c_exam = 2

for c_nurse in [1, 2, 3]:
    rho_reg   = lam / (c_reg   * mu_reg)
    rho_nurse = lam / (c_nurse * mu_nurse)
    rho_exam  = lam / (c_exam  * mu_exam)
    bottleneck = max(
        ('Registration', rho_reg),
        ('Triage',       rho_nurse),
        ('Exam',         rho_exam),
        key=lambda x: x[1]
    )
    print(f"c_nurse={c_nurse}: ρ_reg={rho_reg:.2f}  ρ_nurse={rho_nurse:.2f}  "
          f"ρ_exam={rho_exam:.2f}  → bottleneck: {bottleneck[0]} (ρ={bottleneck[1]:.2f})")

print()
print("Study constraint: nurse utilisation > 70% → need ρ_nurse > 0.70")
print("Which c_nurse values satisfy both stability (ρ<1) and utilisation (ρ>0.70)?")

## 7. Complete Assumptions List

A professional conceptual model document requires ≥ 5 explicit assumptions.

In [ ]:
assumptions = [
    "Patient arrivals follow a Poisson process with constant rate λ=5/hr (no time-of-day variation).",
    "All service times are exponentially distributed (memoryless).",
    "No patient abandons the queue (infinite patience).",
    "The clinic starts empty at t=0 (no patients carried over from previous shift).",
    "Exam rooms are interchangeable; any patient uses any available room.",
    "?",   # add at least one more
    "?",   # what about nurse breaks? equipment failures? no-shows?
]

print("Assumptions and their consequences if violated:")
for i, a in enumerate(assumptions, 1):
    print(f"  {i}. {a}")
    if a != '?':
        print(f"     → Violation would [your answer here]")
    print()

---
## Try It Yourself

1. **Boundary extension**: If you add the pharmacy to the model (patients pick up prescriptions after seeing the physician), which components of the conceptual model change? List new entities, resources, events, and state variables.

2. **Priority discipline**: Suppose patients with acuity ≥ 4 skip the triage queue and go directly to an exam room. Rewrite the event list to handle this routing. What new state variable is needed?

3. **Pre-simulation validation**: The throughput analysis above shows that with c_nurse=1, ρ_nurse > 1 means the model is unstable (queue grows without bound). Verify this claim: if λ=5/hr and μ_nurse=7.5/hr but c_nurse=1, the maximum throughput is 7.5/hr which is ≥ 5/hr. So why is c_nurse=1 actually OK? *Hint: check the numbers again carefully.*